# XGBoost v5 -- per-type sub-models (top-10) + n_est=5000 + log1p target + geo hierarchy

Final notebook for the **xgboost_v5** family from the `model-family-mae` workflow on branch `mlchallenge/may5`. Family is **parked, not promoted** -- it ties v4 W''''' within fold noise on the 2-seed mean (-203 MAE, well below the ~1100 noise band) and does not meet the user-requested "by far" threshold. v4 W''''' remains the global champion, but this notebook captures v5's iter10 recipe for blending and reproducibility.

**Recipe** (deepen iter10 winner = v4 W''''' anchor + 2 kept knobs from v5):
- `XGBRegressor(objective="reg:absoluteerror", n_estimators=5000, learning_rate=0.05, max_depth=6, min_child_weight=5, subsample=0.8, colsample_bytree=0.6, reg_lambda=1.0, tree_method="hist", enable_categorical=True)` -- directly optimizes MAE
- **Target**: `log1p(y)` with `expm1` inverse for predictions
- **Architecture**: 0.7 * global model + 0.3 * per-type sub-model (top 10 property types with min 200 train rows each, dynamically chosen per fold)
- **Numeric features**: full v2 set + `built_per_premise`, `land_per_lot`, `commercial_share`, `apt_share`, `houses_per_premise`, `built_per_land` = `built_area / land_area`, `built_rel_type` = `built_area / mean(built_area | property_type)` (fold-safe group statistic)
- `date_ordinal` = days since 2014-01-01
- **Six native XGBoost categoricals** (`enable_categorical=True`): `commune_first` (first token of `commune_codes`), `cadastral_first` (first token of `cadastral_sections`), `property_type`, `transaction_type`, `dept_code`, `region_code`
- Categorical levels and group statistics fitted on training only; unseen test values fall back to NaN / global mean

**Performance** (5-fold CV-MAE, full audit in `experiments/xgboost_v5/results.tsv` and `iteration_log.md`):

| Stage | Metric | Value |
|---|---|---|
| xgboost_v4 W''''' (current global champion, seed=42) | 5-fold CV-MAE | 47,107.53 |
| xgboost_v4 W''''' confirm (seed=2026) | 5-fold CV-MAE | 47,998.49 |
| xgboost_v4 W''''' 2-seed mean | 5-fold CV-MAE | **47,553.01** |
| **v5 iter10 (this notebook, seed=42)** | **5-fold CV-MAE** | **46,909.14** |
| v5 iter10 confirm (seed=2026) | 5-fold CV-MAE | 47,790.96 |
| **v5 iter10 2-seed mean** | **5-fold CV-MAE** | **47,350.05** |

Delta vs v4 2-seed mean: **-203 MAE** -- a **tie within fold noise** (pooled noise band ~1,100). v5 is recorded as a confirmed within-noise tie usable for blending; v4 stays as the canonical promoted champion.

In [ ]:
import json
import re
import zipfile
from pathlib import Path

import dagshub
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, train_test_split
from xgboost import XGBRegressor

dagshub.init(repo_owner="abdul.aldalali", repo_name="ML_Challenge_ML_Class_DSS", mlflow=True)
mlflow.set_experiment("xgboost-v5")

cwd = Path.cwd()
if (cwd / "dataset").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "dataset").exists():
    PROJECT_ROOT = cwd.parent
elif (cwd.parent.parent / "dataset").exists():
    PROJECT_ROOT = cwd.parent.parent
else:
    raise FileNotFoundError("Could not find project root containing dataset/")

EXPERIMENT_NAME = "xgboost_v5"
EXPERIMENT_DIR = PROJECT_ROOT / "experiments" / EXPERIMENT_NAME
DATASET_DIR = PROJECT_ROOT / "dataset"

TARGET_KEY = "property_value"
VALIDATION_SIZE = 0.2
RANDOM_STATE = 42
CV_FOLDS = 5

# Feature configuration -- mirrors experiments/xgboost_v5/runs/.../experiment.py
DATE_EPOCH = pd.Timestamp("2014-01-01")
GEO_MULTIVALUE_COLUMNS = ("commune_codes", "cadastral_sections")
CATEGORICAL_FEATURES = (
    "property_type", "commune_first", "cadastral_first", "transaction_type",
    "dept_code", "region_code",
)
NUMERIC_COLUMNS = (
    "built_area", "house_area", "area_house_5plus_rooms", "land_area",
    "num_lots", "num_premises", "num_houses", "num_apartments", "num_commercial",
    "apartment_area", "num_house_5plus_rooms",
    "area_house_4_rooms", "num_house_4_rooms",
    "area_house_3_rooms", "num_house_3_rooms",
    "area_house_2_rooms", "num_house_2_rooms",
    "area_house_1_room", "num_house_1_room",
    "area_apt_5plus_rooms", "num_apt_5plus_rooms",
    "area_apt_4_rooms", "num_apt_4_rooms",
    "area_apt_3_rooms", "num_apt_3_rooms",
    "area_apt_2_rooms", "num_apt_2_rooms",
    "area_apt_1_room", "num_apt_1_room",
    "date_ordinal",
    "built_per_premise", "land_per_lot", "commercial_share", "apt_share", "houses_per_premise",
    "built_per_land",
    "built_rel_type",
)
DERIVED_RATIO_COLUMNS = (
    "built_per_premise", "land_per_lot", "commercial_share", "apt_share",
    "houses_per_premise", "built_per_land", "built_rel_type",
)
DATE_FEATURE_COLUMNS = ("date_ordinal",)

# Per-type sub-model architecture
SUB_MODEL_TOP_K = 10
SUB_MODEL_MIN_COUNT = 200
SUB_MODEL_BLEND = 0.3  # final = (1-BLEND)*global + BLEND*sub

# XGBoost hyperparameters
XGB_PARAMS = dict(
    objective="reg:absoluteerror",
    n_estimators=5000,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.6,
    reg_lambda=1.0,
    tree_method="hist",
    n_jobs=-1,
    random_state=RANDOM_STATE,
    enable_categorical=True,
)

USE_LOG_TARGET = True

TRAIN_PATH = DATASET_DIR / "train.json"
TEST_PATH = DATASET_DIR / "test.json"
PREDICTION_JSON_PATH = EXPERIMENT_DIR / "predicted.json"
PREDICTION_ZIP_PATH = EXPERIMENT_DIR / "predicted.zip"

In [ ]:
def load_json(path):
    """Load a JSON file and return the parsed Python object."""
    with open(path, "r", encoding="utf-8") as handle:
        return json.load(handle)


def make_target_array(records):
    return np.array([record[TARGET_KEY] for record in records], dtype=np.float64)


def first_token(value):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return np.nan
    text = str(value).strip()
    if not text:
        return np.nan
    return text.split(",")[0].strip()


def add_geo_first_tokens(frame):
    frame = frame.copy()
    if "commune_codes" in frame.columns:
        frame["commune_first"] = frame["commune_codes"].map(first_token)
    if "cadastral_sections" in frame.columns:
        frame["cadastral_first"] = frame["cadastral_sections"].map(first_token)
    return frame


def add_date_ordinal(frame):
    frame = frame.copy()
    if "transaction_date" in frame.columns:
        first_date = frame["transaction_date"].astype(str).str.split(",").str[0].str.strip()
        parsed = pd.to_datetime(first_date, errors="coerce")
        frame["date_ordinal"] = (parsed - DATE_EPOCH).dt.days.astype("Float64")
    else:
        frame["date_ordinal"] = np.nan
    return frame


def safe_div(numerator, denominator):
    num = pd.to_numeric(numerator, errors="coerce")
    den = pd.to_numeric(denominator, errors="coerce")
    den = den.where(den > 0, np.nan)
    return num / den


def add_derived_ratios(frame):
    frame = frame.copy()
    frame["built_per_premise"] = safe_div(frame.get("built_area"), frame.get("num_premises"))
    frame["land_per_lot"] = safe_div(frame.get("land_area"), frame.get("num_lots"))
    frame["commercial_share"] = safe_div(frame.get("num_commercial"), frame.get("num_premises"))
    frame["apt_share"] = safe_div(frame.get("num_apartments"), frame.get("num_premises"))
    frame["houses_per_premise"] = safe_div(frame.get("num_houses"), frame.get("num_premises"))
    frame["built_per_land"] = safe_div(frame.get("built_area"), frame.get("land_area"))
    return frame


def compute_built_rel_type_map(train_frame):
    """Fold-safe: compute mean(built_area | property_type) on TRAIN ONLY."""
    df = pd.DataFrame({
        "type": train_frame["property_type"].astype(str),
        "built": pd.to_numeric(train_frame.get("built_area"), errors="coerce"),
    })
    df = df[(df["type"] != "nan") & df["built"].notna() & (df["built"] > 0)]
    means = df.groupby("type")["built"].mean()
    global_mean = df["built"].mean()
    return means.to_dict(), global_mean


def add_built_rel_type(frame, type_means, global_mean):
    frame = frame.copy()
    type_str = frame["property_type"].astype(str)
    type_mean_per_row = type_str.map(type_means).fillna(global_mean).astype(float)
    built = pd.to_numeric(frame.get("built_area"), errors="coerce")
    frame["built_rel_type"] = built / type_mean_per_row.replace(0, np.nan)
    return frame


def make_feature_frame_pre(records):
    """Apply all preprocessing EXCEPT built_rel_type (which needs train-fold means)."""
    return add_derived_ratios(add_date_ordinal(add_geo_first_tokens(pd.DataFrame(records))))


def materialize_features(frame, category_levels):
    numeric_part = (
        frame.reindex(columns=list(NUMERIC_COLUMNS))
        .apply(pd.to_numeric, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
    )
    cat_part = pd.DataFrame(index=frame.index)
    for column in CATEGORICAL_FEATURES:
        levels = category_levels[column]
        series = frame[column].astype(str) if column in frame.columns else pd.Series([np.nan] * len(frame), index=frame.index)
        cat_part[column] = pd.Categorical(series, categories=levels)
    return pd.concat([numeric_part, cat_part], axis=1)


def fit_category_levels(records):
    frame = add_geo_first_tokens(pd.DataFrame(records))
    levels = {}
    for column in CATEGORICAL_FEATURES:
        if column in frame.columns:
            levels[column] = list(pd.Series(frame[column].astype(str)).dropna().unique())
        else:
            levels[column] = []
    return levels


def top_k_types(records, k=SUB_MODEL_TOP_K, min_count=SUB_MODEL_MIN_COUNT):
    series = pd.DataFrame(records).get("property_type", pd.Series([], dtype=object)).astype(str)
    counts = series[series != "nan"].value_counts()
    counts = counts[counts >= min_count]
    return list(counts.head(k).index)


def build_model():
    return XGBRegressor(**XGB_PARAMS)


def fit_predict(train_records, validation_records):
    """One CV-fold or final-train evaluator: implements v5 iter10 recipe exactly.

    Mirrors experiments/xgboost_v5/runs/20260508-160016-deepen-v4_anchor/experiment.py
    so MLflow-logged predictions match the workflow's canonical CV scores.
    """
    category_levels = fit_category_levels(train_records)

    train_pre = make_feature_frame_pre(train_records)
    val_pre = make_feature_frame_pre(validation_records)

    type_means, global_mean = compute_built_rel_type_map(train_pre)
    train_pre = add_built_rel_type(train_pre, type_means, global_mean)
    val_pre = add_built_rel_type(val_pre, type_means, global_mean)

    train_frame = materialize_features(train_pre, category_levels)
    val_frame = materialize_features(val_pre, category_levels)

    y_train = make_target_array(train_records)
    y_train_log = np.log1p(y_train) if USE_LOG_TARGET else y_train

    global_model = build_model()
    global_model.fit(train_frame, y_train_log)
    pred_global_log = global_model.predict(val_frame)
    pred_global = np.expm1(pred_global_log) if USE_LOG_TARGET else pred_global_log

    train_types = pd.DataFrame(train_records).get(
        "property_type", pd.Series([""] * len(train_records))
    ).astype(str).values
    val_types = pd.DataFrame(validation_records).get(
        "property_type", pd.Series([""] * len(validation_records))
    ).astype(str).values

    final_pred = pred_global.copy()
    sub_types = top_k_types(train_records)
    sub_models = {}
    for sub_type in sub_types:
        train_mask = train_types == sub_type
        val_mask = val_types == sub_type
        if train_mask.sum() < 100 or val_mask.sum() == 0:
            continue
        sub_train_frame = train_frame[train_mask]
        sub_y = y_train_log[train_mask]
        sub_model = build_model()
        sub_model.fit(sub_train_frame, sub_y)
        sub_pred_log = sub_model.predict(val_frame[val_mask])
        sub_pred = np.expm1(sub_pred_log) if USE_LOG_TARGET else sub_pred_log
        final_pred[val_mask] = (
            (1 - SUB_MODEL_BLEND) * pred_global[val_mask]
            + SUB_MODEL_BLEND * sub_pred
        )
        sub_models[sub_type] = sub_model
    return final_pred, global_model, sub_models, sub_types

In [ ]:
train_records = load_json(TRAIN_PATH)
test_records = load_json(TEST_PATH)

train_split_records, validation_records = train_test_split(
    train_records,
    test_size=VALIDATION_SIZE,
    random_state=RANDOM_STATE,
)

with mlflow.start_run(run_name=EXPERIMENT_NAME):
    # Params
    mlflow.log_param("model_type", "XGBRegressor")
    mlflow.log_param("recipe", "v5_iter10_top10_sub_models_nest5000")
    mlflow.log_param("anchor", "xgboost_v4_W_5primes")
    mlflow.log_param("validation_size", VALIDATION_SIZE)
    mlflow.log_param("random_state", RANDOM_STATE)
    mlflow.log_param("cv_folds", CV_FOLDS)
    for k, v in XGB_PARAMS.items():
        mlflow.log_param(k, v)
    mlflow.log_param("target", TARGET_KEY)
    mlflow.log_param("use_log_target", USE_LOG_TARGET)
    mlflow.log_param("sub_model_top_k", SUB_MODEL_TOP_K)
    mlflow.log_param("sub_model_min_count", SUB_MODEL_MIN_COUNT)
    mlflow.log_param("sub_model_blend", SUB_MODEL_BLEND)
    mlflow.log_param("numeric_features", ",".join(NUMERIC_COLUMNS))
    mlflow.log_param("categorical_features", ",".join(CATEGORICAL_FEATURES))
    mlflow.log_param("derived_ratios", ",".join(DERIVED_RATIO_COLUMNS))
    mlflow.log_param("date_features", ",".join(DATE_FEATURE_COLUMNS))

    # Sizes
    mlflow.log_metric("train_records", len(train_records))
    mlflow.log_metric("train_split_records", len(train_split_records))
    mlflow.log_metric("validation_records", len(validation_records))
    mlflow.log_metric("test_records", len(test_records))
    mlflow.log_metric("numeric_feature_count", len(NUMERIC_COLUMNS))
    mlflow.log_metric("categorical_feature_count", len(CATEGORICAL_FEATURES))

    # 80/20 holdout (matches HGB / xgboost notebook for cross-family comparability)
    print("Fitting 80/20 holdout...")
    holdout_pred, _, _, _ = fit_predict(train_split_records, validation_records)
    validation_targets = make_target_array(validation_records)
    validation_mae = mean_absolute_error(validation_targets, holdout_pred)
    validation_rmse = float(np.sqrt(mean_squared_error(validation_targets, holdout_pred)))
    validation_r2 = r2_score(validation_targets, holdout_pred)

    mlflow.log_metric("validation_mae", validation_mae)
    mlflow.log_metric("validation_rmse", validation_rmse)
    mlflow.log_metric("validation_r2", validation_r2)

    # 5-fold CV-MAE -- canonical workflow metric (matches scripts/eval_family.py)
    print("Running 5-fold CV...")
    kf = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    fold_maes = []
    for fold, (tr_idx, vl_idx) in enumerate(kf.split(np.arange(len(train_records)))):
        fold_train = [train_records[i] for i in tr_idx]
        fold_val = [train_records[i] for i in vl_idx]
        fold_pred, _, _, _ = fit_predict(fold_train, fold_val)
        fold_mae = mean_absolute_error(make_target_array(fold_val), fold_pred)
        fold_maes.append(fold_mae)
        mlflow.log_metric(f"cv_fold_{fold}_mae", fold_mae)
        print(f"  fold_{fold}_mae: {fold_mae:,.4f}")

    cv_mae = float(np.mean(fold_maes))
    cv_mae_std = float(np.std(fold_maes, ddof=0))
    mlflow.log_metric("cv_mae", cv_mae)
    mlflow.log_metric("cv_mae_std", cv_mae_std)

    # Final: train on full data, predict test set
    print("Training final model on full data and predicting test set...")
    test_pred, final_global_model, final_sub_models, final_sub_types = fit_predict(
        train_records, test_records
    )
    predictions = [{TARGET_KEY: float(value)} for value in test_pred]

    PREDICTION_JSON_PATH.parent.mkdir(parents=True, exist_ok=True)
    with open(PREDICTION_JSON_PATH, "w", encoding="utf-8") as handle:
        json.dump(predictions, handle, indent=2)
    with zipfile.ZipFile(PREDICTION_ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        archive.write(PREDICTION_JSON_PATH, arcname="predicted.json")

    mlflow.log_metric("prediction_count", len(predictions))
    mlflow.log_param("final_sub_types", ",".join(final_sub_types))
    mlflow.log_metric("final_sub_model_count", len(final_sub_models))

    # Log the global model (sub-models are mirrored as separate artifacts).
    # MLflow 3 validates `name` server-side; keep names simple and unique per run.
    mlflow.sklearn.log_model(final_global_model, name="global_model")
    for i, (sub_type, sub_model) in enumerate(final_sub_models.items()):
        safe_name = re.sub(r"[^A-Za-z0-9_-]+", "_", sub_type).strip("_")[:50] or "type"
        mlflow.sklearn.log_model(sub_model, name=f"sub_{i:02d}_{safe_name}")

    mlflow.log_artifact(str(PREDICTION_JSON_PATH))
    mlflow.log_artifact(str(PREDICTION_ZIP_PATH))

print()
print(f"Validation MAE (holdout):  {validation_mae:,.2f}")
print(f"Validation RMSE (holdout): {validation_rmse:,.2f}")
print(f"Validation R2 (holdout):   {validation_r2:.4f}")
print(f"5-fold CV MAE:             {cv_mae:,.2f} (std {cv_mae_std:,.2f})")
print(f"Wrote {len(predictions)} predictions to {PREDICTION_ZIP_PATH}")

In [ ]:
{
    "experiment": EXPERIMENT_NAME,
    "primary_metric": "cv_mae",
    "cv_mae": cv_mae,
    "cv_mae_std": cv_mae_std,
    "validation_mae": validation_mae,
    "validation_rmse": validation_rmse,
    "validation_r2": validation_r2,
    "numeric_feature_count": len(NUMERIC_COLUMNS),
    "categorical_features": list(CATEGORICAL_FEATURES),
    "sub_model_top_k": SUB_MODEL_TOP_K,
    "sub_model_blend": SUB_MODEL_BLEND,
    "use_log_target": USE_LOG_TARGET,
    "n_estimators": XGB_PARAMS["n_estimators"],
    "prediction_json": str(PREDICTION_JSON_PATH),
    "prediction_zip": str(PREDICTION_ZIP_PATH),
}